In [ ]:
%%writefile qa_chatbot_app.py
import streamlit as st
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, HumanMessage
from langsmith import traceable  # optional

# Load environment variables
load_dotenv()
gpk = os.getenv("GPK")
langsmith_api_key = os.getenv("LANGCHAIN_API_KEY")
project_name = os.getenv("LANGCHAIN_PROJECT", "Groq_ChatBot_Project")

# LangSmith setup
if langsmith_api_key:
    from langchain.callbacks.tracers.langchain import LangChainTracer
    tracer = LangChainTracer(project_name=project_name)
else:
    tracer = None

# Streamlit UI
st.set_page_config(page_title="Groq Chatbot", page_icon="🤖")
st.title("🤖 Q&A Chatbot using Groq + LangChain + LangSmith")

# Sidebar configs
temperature = st.sidebar.slider("Temperature", 0.0, 1.0, 0.7)
max_tokens = st.sidebar.slider("Max Tokens", 50, 1024, 300)
model_name = st.sidebar.selectbox("Choose Model", ["gemma2-9b-it", "llama-3.1-8b-instant", "llama-3.3-70b-versatile"], index=0)

# Memory setup
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# Prompt template with memory
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{question}")
])

llm = ChatGroq(
    model=model_name,
    groq_api_key=gpk,
    temperature=temperature,
    max_tokens=max_tokens
)

output_parser = StrOutputParser()

chain: Runnable = prompt | llm | output_parser

@traceable(name="generate_response", project_name=project_name)
def generate_response(question: str):
    inputs = {
        "question": question,
        "chat_history": st.session_state.chat_history
    }
    if tracer:
        return chain.invoke(inputs, config={"callbacks": [tracer]})
    return chain.invoke(inputs)

# Display existing chat
for msg in st.session_state.chat_history:
    if isinstance(msg, HumanMessage):
        st.chat_message("user").write(msg.content)
    elif isinstance(msg, AIMessage):
        st.chat_message("assistant").write(msg.content)

# Chat input
user_query = st.chat_input("Ask your question")
if user_query:
    st.chat_message("user").write(user_query)
    st.session_state.chat_history.append(HumanMessage(content=user_query))
    
    with st.spinner("Generating response..."):
        try:
            response = generate_response(user_query)
            st.chat_message("assistant").write(response)
            st.session_state.chat_history.append(AIMessage(content=response))
        except Exception as e:
            st.error(f"Error: {e}")


In [ ]:
!streamlit run qa_chatbot_app.py

# Introduction To Groq Cloud And LPU Inference Engine

In [21]:
#pip install faiss-cpu
#!pip install pypdf

Defaulting to user installation because normal site-packages is not writeable


In [26]:
%%writefile groq_api.py

import streamlit as st
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFDirectoryLoader
import sentence_transformers
import time
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Load environment variables
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize LLM
llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.3-70b-versatile")

# Prompt template
prompt = ChatPromptTemplate.from_template(
    """
    Answer the questions based on the provided context only.
    Please provide the most accurate response based on the question.

    <context>
    {context}
    </context>

    Question: {input}
    """
)

# Vector creation function with debugging
def create_vs():
    if "vectors" not in st.session_state:
        st.write("Loading and processing documents...")

        try:
            # Load embeddings model
            embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
            st.session_state.embeddings = embedding_model

            # Load PDF documents
            #base_dir = os.path.dirname(os.path.abspath(__file__))
            #pdf_dir = os.path.join(base_dir, "papers")
            #st.session_state.loader = PyPDFDirectoryLoader(pdf_dir)
            st.session_state.loader = DirectoryLoader(
                "C:/Users/nithi/Desktop/GitHub/LLM/GEN AI APP/papers",
                glob="**/*.pdf",
                loader_cls=PyPDFLoader
            )
            st.session_state.docs = st.session_state.loader.load()
            st.write(f"✅ Loaded {len(st.session_state.docs)} documents")
            for i, doc in enumerate(st.session_state.docs):
                st.write(f"📄 Document {i+1}: {doc.metadata.get('source', 'Unknown')} - {len(doc.page_content)} characters")

            if not st.session_state.docs:
                st.error("❌ No documents loaded. Check the /papers directory.")
                return

            # Split documents
            st.session_state.text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
            st.session_state.final_documents = st.session_state.text_splitter.split_documents(st.session_state.docs)
            st.write(f"✅ Split into {len(st.session_state.final_documents)} text chunks")

            # Test embedding generation
            test_text = st.session_state.final_documents[0].page_content
            emb_test = st.session_state.embeddings.embed_query(test_text)
            st.write(f"✅ Sample embedding length: {len(emb_test)}")

            # Create vector store
            st.session_state.vectors = FAISS.from_documents(
                st.session_state.final_documents,
                st.session_state.embeddings
            )
            st.success("✅ Vector database is ready")

        except Exception as e:
            st.error(f"❌ Error during vector creation: {e}")

# Streamlit UI
st.title("GROQ PDF Q&A App")

user_prompts = st.text_input("Enter your query from the papers")

if st.button("Document Embedding"):
    create_vs()

if user_prompts:
    if "vectors" not in st.session_state:
        st.warning("⚠️ Please create the vector store first by clicking the 'Document Embedding' button.")
    else:
        try:
            document_chain = create_stuff_documents_chain(llm, prompt)
            retriever = st.session_state.vectors.as_retriever()
            ret_chain = create_retrieval_chain(retriever, document_chain)

            start = time.process_time()
            response = ret_chain.invoke({'input': user_prompts})
            duration = time.process_time() - start

            st.write(f"⏱️ Response Time: {duration:.2f} seconds")
            st.write("💬 Answer:", response['answer'])

            with st.expander("🧠 Document Similarity Context"):
                for i, doc in enumerate(response.get('context', [])):
                    st.write(f"🔹 Chunk {i+1}")
                    st.write(doc.page_content)
                    st.write("---")

        except Exception as e:
            st.error(f"❌ Error during query processing: {e}")


Overwriting groq_api.py


In [ ]:
!streamlit run groq_api.py